Шаг 1: Установка и загрузка библиотек

In [46]:
!pip install spacy
!python -m spacy download ru_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 31.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [47]:
import re
import math
import spacy
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
nlp = spacy.load('ru_core_news_sm')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Шаг 2: Исходные данные (Корпус текстов)

In [113]:
corpus = [
    "Сегодня была отличная погода, и мы пошли гулять в красивый парк!",
    "Ужасная погода... Весь день идет сильный дождь, прогулка отменяется.",
    "В парке мы видели красивых птиц. Погода шепчет, солнце светит ярко.",
    "Дождь не прекращается! Сильный ветер сорвал все планы на вечер."
]

sample_text = corpus[0]
print("Исходный текст для анализа:", sample_text)

Исходный текст для анализа: Сегодня была отличная погода, и мы пошли гулять в красивый парк!


Шаг 3: Очистка текста с помощью Регулярных выражений (RegEx)

In [103]:
text_lowercase = sample_text.lower()
text_cleaned = re.sub(r'[^a-яё\s]', '', text_lowercase)

print("Текст после очистки RegEx:")
print(text_cleaned)

Текст после очистки RegEx:
сегодня была отличная погода и мы пошли гулять в красивый парк


Шаг 4: Токенизация, удаление стоп-слов и Лемматизация (spaCy)

In [101]:
russian_stopwords = set(stopwords.words('russian'))
doc = nlp(text_cleaned)

filtered_tokens = []
for token in doc:
  lemma = token.lemma_
  if lemma not in russian_stopwords and lemma.strip():
    filtered_tokens.append(lemma)

print("Токены после лемматизации и удаления стоп-слов:")
print(filtered_tokens)

Токены после лемматизации и удаления стоп-слов:
['сегодня', 'отличный', 'погода', 'пойти', 'гулять', 'красивый', 'парк']


Шаг 5: Полная предобработка всего текста

In [119]:
def pipeline_preprocess(text):
  text = text.lower()
  text = re.sub(r'[^а-яё\s]', '', text)
  doc = nlp(text)
  return [token.lemma_ for token in doc if token.lemma_ not in russian_stopwords and token.lemma_.strip()]

processed_corpus = [pipeline_preprocess(text) for text in corpus]

for i, doc_tokens in enumerate(processed_corpus):
  print(f"Документ {i+1}: {doc_tokens}")

Документ 1: ['отличный', 'погода', 'пойти', 'гулять', 'красивый', 'парк']
Документ 2: ['ужасный', 'погода', 'день', 'идти', 'сильный', 'дождь', 'прогулка', 'отменяться']
Документ 3: ['парк', 'видеть', 'красивый', 'птица', 'погода', 'шепчет', 'солнце', 'светить', 'ярко']
Документ 4: ['дождь', 'прекращаться', 'сильный', 'ветер', 'сорвать', 'план', 'вечер']
Документ 5: ['купить', 'новый', 'зонт', 'улица', 'идти', 'сильный', 'дождь']


Шаг 6: Векторизация текста (Расчет TF-IDF вручную)

In [120]:
vocabulary = sorted(list(set([word for doc in processed_corpus for word in doc])))
print("Уникальный словрь корпуса (всего слов:", len(vocabulary), "):")
print(vocabulary)

def calculate_tfidf(word, current_doc, all_docs):
  tf = current_doc.count(word) / len(current_doc) if len(current_doc) > 0 else 0
  docs_with_word = sum(1 for d in all_docs if word in d)
  idf = math.log(len(all_docs) / docs_with_word)

  return round(tf * idf, 4)

print("\nВекторы документов (TF-IDF числовые представления):")
for i, doc in enumerate(processed_corpus):
  vector = []
  for word in vocabulary:
    tfidf_value = calculate_tfidf(word, doc, processed_corpus)
    vector.append(tfidf_value)
  print(f"Вектор документа {i+1}: {vector}")

Уникальный словрь корпуса (всего слов: 28 ):
['ветер', 'вечер', 'видеть', 'гулять', 'день', 'дождь', 'зонт', 'идти', 'красивый', 'купить', 'новый', 'отличный', 'отменяться', 'парк', 'план', 'погода', 'пойти', 'прекращаться', 'прогулка', 'птица', 'светить', 'сильный', 'солнце', 'сорвать', 'ужасный', 'улица', 'шепчет', 'ярко']

Векторы документов (TF-IDF числовые представления):
Вектор документа 1: [0.0, 0.0, 0.0, 0.2682, 0.0, 0.0, 0.0, 0.0, 0.1527, 0.0, 0.0, 0.2682, 0.0, 0.1527, 0.0, 0.0851, 0.2682, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Вектор документа 2: [0.0, 0.0, 0.0, 0.0, 0.2012, 0.0639, 0.0, 0.1145, 0.0, 0.0, 0.0, 0.0, 0.2012, 0.0, 0.0, 0.0639, 0.0, 0.0, 0.2012, 0.0, 0.0, 0.0639, 0.0, 0.0, 0.2012, 0.0, 0.0, 0.0]
Вектор документа 3: [0.0, 0.0, 0.1788, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1018, 0.0, 0.0, 0.0, 0.0, 0.1018, 0.0, 0.0568, 0.0, 0.0, 0.0, 0.1788, 0.1788, 0.0, 0.1788, 0.0, 0.0, 0.0, 0.1788, 0.1788]
Вектор документа 4: [0.2299, 0.2299, 0.0, 0.0, 0.0, 0.073, 0.0, 0.0,

Задание 1

In [55]:
print("Общее количество слов в списке:", len(russian_stopwords))
print("Список в алфовитном порядке:", sorted(russian_stopwords))

Общее количество слов в списке: 153
Список в алфовитном порядке: ['а', 'без', 'более', 'больше', 'будет', 'будто', 'бы', 'был', 'была', 'были', 'было', 'быть', 'в', 'вам', 'вас', 'вдруг', 'ведь', 'весь', 'во', 'вот', 'впрочем', 'все', 'всегда', 'всего', 'всех', 'всю', 'вы', 'где', 'да', 'даже', 'два', 'для', 'до', 'другой', 'его', 'ее', 'ей', 'ему', 'если', 'есть', 'еще', 'ж', 'же', 'за', 'зачем', 'здесь', 'и', 'из', 'или', 'им', 'иногда', 'их', 'к', 'как', 'какая', 'какой', 'когда', 'конечно', 'кто', 'куда', 'ли', 'лучше', 'между', 'меня', 'мне', 'много', 'может', 'можно', 'мой', 'моя', 'мы', 'на', 'над', 'надо', 'наконец', 'нас', 'не', 'него', 'нее', 'ней', 'нельзя', 'нет', 'ни', 'нибудь', 'никогда', 'ним', 'них', 'ничего', 'но', 'ну', 'о', 'об', 'один', 'он', 'она', 'они', 'опять', 'от', 'перед', 'по', 'под', 'после', 'потом', 'потому', 'почти', 'при', 'про', 'раз', 'разве', 'с', 'сам', 'свою', 'себе', 'себя', 'сегодня', 'сейчас', 'со', 'совсем', 'так', 'такой', 'там', 'тебя', 'тем'

In [116]:
russian_stopwords.update(['сегодня', 'весь'])
processed_corpus = [pipeline_preprocess(text) for text in corpus]

print(f"Документ 1: {processed_corpus[0]}")

Документ 1: ['отличный', 'погода', 'пойти', 'гулять', 'красивый', 'парк']


Задание 2

In [117]:
corpus.append("Сейчас я куплю новый зонт, потому что на улице идёт сильный дождь.")
for doc in corpus:
  print(f"Документ {corpus.index(doc) + 1}: {doc}")

Документ 1: Сегодня была отличная погода, и мы пошли гулять в красивый парк!
Документ 2: Ужасная погода... Весь день идет сильный дождь, прогулка отменяется.
Документ 3: В парке мы видели красивых птиц. Погода шепчет, солнце светит ярко.
Документ 4: Дождь не прекращается! Сильный ветер сорвал все планы на вечер.
Документ 5: Сейчас я куплю новый зонт, потому что на улице идёт сильный дождь.


Задание 3

In [126]:
corpus = [
    "Сегодня была отличная погода, и мы пошли гулять в красивый парк!",
    "Ужасная погода... Весь день идет сильный дождь, прогулка отменяется.",
    "В 2026 году, в парке мы видели красивых птиц. Погода шепчет, солнце светит ярко.",
    "Дождь не прекращается! Сильный ветер сорвал все планы на вечер.",
    "Сейчас я куплю новый зонт, потому что на улице идёт сильный дождь."
]

def pipeline_preprocess(text):
  text = text.lower()
  text = re.sub(r'[^а-яё0-9\s]', '', text)
  doc = nlp(text)
  return [token.lemma_ for token in doc if token.lemma_ not in russian_stopwords and token.lemma_.strip()]

In [127]:
processed_corpus = [pipeline_preprocess(text) for text in corpus]
print(processed_corpus[2])

['2026', 'год', 'парк', 'видеть', 'красивый', 'птица', 'погода', 'шепчет', 'солнце', 'светить', 'ярко']
